# AutoPET-III Preprocessing (CPU-only)

Phase 1 of the split-architecture segmentation pipeline. Builds paired CT+PET NIfTI inputs for every case and writes them to Drive at `/My Drive/P79 Data/autopet_iii/paired_inputs/{pt_uid}_{0000|0001}.nii.gz`.

**Run this on a CPU runtime (free quota).** The GPU sits idle during preprocessing in the original `segment_autopet_iii.ipynb`, so splitting the pipeline lets the CPU phase run on the free CPU runtime and the GPU phase consume only inference time.

**Resume invariant:** a case is considered done if both `_0000.nii.gz` and `_0001.nii.gz` exist on Drive. Re-running this notebook skips done cases.

**Storage cost:** ~60-120 GB on Drive (paired NIfTIs are ~100-200 MB per case for 597 cases). The pairs can be deleted after inference completes — they are intermediate artifacts only.

**Pre-registration:** https://doi.org/10.17605/OSF.IO/4KAZN (§5.2.2)

In [ ]:
# Step 1: Mount Drive, check disk
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
free = shutil.disk_usage('/content/drive/MyDrive').free / (1024**3)
print(f'Drive free: {free:.1f} GB (need ~60-120 GB for paired NIfTIs)')

!pip install -q pydicom SimpleITK nibabel tcia_utils

In [ ]:
# Step 2: Upload suv_conversion.py from project source
import sys
from google.colab import files

target = '/content/suv_conversion.py'
if os.path.exists(target):
    os.remove(target)

uploaded = files.upload()
assert 'suv_conversion.py' in uploaded, 're-run and select suv_conversion.py'

sys.path.insert(0, '/content')
sys.modules.pop('suv_conversion', None)
from suv_conversion import extract_pet_metadata, dicom_series_to_suv_sitk
print('Imported suv_conversion helpers')

In [ ]:
# Step 3: Load PT/CT pair manifest from Drive (built previously by segment_autopet_iii.ipynb)
import pandas as pd

DRIVE_ROOT = '/content/drive/MyDrive/P79 Data/autopet_iii'
MANIFEST_PATH = f'{DRIVE_ROOT}/_pt_ct_pairs.csv'

if not os.path.exists(MANIFEST_PATH):
    # Fall back: rebuild from TCIA
    from tcia_utils import nbia
    series_df = nbia.getSeries(collection='PSMA-PET-CT-Lesions', format='df')
    pt = series_df[series_df['Modality'] == 'PT'][['SeriesInstanceUID','StudyInstanceUID']].rename(
        columns={'SeriesInstanceUID':'pt_uid'})
    ct = series_df[series_df['Modality'] == 'CT'][['SeriesInstanceUID','StudyInstanceUID']].rename(
        columns={'SeriesInstanceUID':'ct_uid'})
    pairs = pt.merge(ct, on='StudyInstanceUID', how='inner')
    drive_entries = set(os.listdir(DRIVE_ROOT))
    on_drive = lambda u: f'{u}.zip' in drive_entries or u in drive_entries
    ready = pairs[pairs['pt_uid'].apply(on_drive) & pairs['ct_uid'].apply(on_drive)].copy()
    ready[['StudyInstanceUID','pt_uid','ct_uid']].to_csv(MANIFEST_PATH, index=False)

manifest = pd.read_csv(MANIFEST_PATH)
print(f'Manifest: {len(manifest)} PT/CT pairs')

In [ ]:
# Step 4: Helpers (DICOM loader, CT-to-PET resampler, paired-NIfTI builder)
import zipfile, tempfile, glob
import numpy as np
import SimpleITK as sitk
import pydicom

def load_dicom_series(series_uid, drive_root=DRIVE_ROOT):
    zip_path = os.path.join(drive_root, f'{series_uid}.zip')
    dir_path = os.path.join(drive_root, series_uid)
    if os.path.exists(zip_path):
        tmp_dir = tempfile.mkdtemp(prefix='dcmser_')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(tmp_dir)
        entries = os.listdir(tmp_dir)
        if len(entries) == 1 and os.path.isdir(os.path.join(tmp_dir, entries[0])):
            inner = os.path.join(tmp_dir, entries[0])
        else:
            inner = tmp_dir
        return inner, lambda: shutil.rmtree(tmp_dir, ignore_errors=True)
    elif os.path.isdir(dir_path):
        return dir_path, lambda: None
    else:
        raise FileNotFoundError(f'{series_uid} not on Drive')

def read_ct_as_hu_sitk(ct_dir):
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(ct_dir)
    if not series_ids:
        raise ValueError(f'No DICOM series in {ct_dir}')
    file_names = reader.GetGDCMSeriesFileNames(ct_dir, series_ids[0])
    reader.SetFileNames(file_names)
    return reader.Execute()

def resample_to_reference(moving_image, reference_image, default_value=-1024.0):
    return sitk.Resample(
        moving_image, reference_image, sitk.Transform(),
        sitk.sitkLinear, default_value, moving_image.GetPixelID(),
    )

def build_paired_niftis(pt_uid, ct_uid, out_dir, case_name=None):
    case = case_name or pt_uid
    os.makedirs(out_dir, exist_ok=True)
    pt_dir, pt_cleanup = load_dicom_series(pt_uid)
    ct_dir, ct_cleanup = load_dicom_series(ct_uid)
    try:
        any_pt_dcm = next(iter(
            glob.glob(os.path.join(pt_dir, '*.dcm')) or
            [os.path.join(pt_dir, e) for e in os.listdir(pt_dir)
             if os.path.isfile(os.path.join(pt_dir, e))]
        ))
        meta = extract_pet_metadata(any_pt_dcm)
        suv_image = dicom_series_to_suv_sitk(pt_dir, meta)
        ct_image = read_ct_as_hu_sitk(ct_dir)
        ct_resampled = resample_to_reference(ct_image, suv_image, default_value=-1024.0)
        sitk.WriteImage(ct_resampled, os.path.join(out_dir, f'{case}_0000.nii.gz'))
        sitk.WriteImage(suv_image,   os.path.join(out_dir, f'{case}_0001.nii.gz'))
        return {
            'case': case,
            'patient_id': meta.patient_id,
            'manufacturer': meta.manufacturer,
            'radionuclide': meta.radionuclide,
            'suv_max': float(np.max(sitk.GetArrayFromImage(suv_image))),
        }
    finally:
        pt_cleanup()
        ct_cleanup()

print('Helpers defined')

In [ ]:
# Step 5: PARALLEL PREPROCESSING -- write paired NIfTIs to Drive
# Resume invariant: skip pt_uid if both _0000.nii.gz and _0001.nii.gz exist on Drive.
# 4 worker threads; Drive supports parallel reads + writes; CPU/RAM are not bottlenecks.
import time, json, threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

NUM_WORKERS = 4
PAIRED_DIR = f'{DRIVE_ROOT}/paired_inputs'
ERROR_LOG = f'{DRIVE_ROOT}/_preprocessing_errors.txt'
HEARTBEAT = f'{DRIVE_ROOT}/_preprocessing_heartbeat.json'
os.makedirs(PAIRED_DIR, exist_ok=True)

print_lock = threading.Lock()
hb_lock = threading.Lock()

def hb(state, **extra):
    payload = {'state': state, 'ts': datetime.now().isoformat(), **extra}
    with hb_lock:
        tmp = HEARTBEAT + '.tmp'
        with open(tmp, 'w') as f: json.dump(payload, f, indent=2)
        os.replace(tmp, HEARTBEAT)

def safe_print(msg):
    with print_lock:
        print(msg, flush=True)

# Determine work
existing_pt = {f[:-len('_0001.nii.gz')] for f in os.listdir(PAIRED_DIR)
               if f.endswith('_0001.nii.gz')}
existing_ct = {f[:-len('_0000.nii.gz')] for f in os.listdir(PAIRED_DIR)
               if f.endswith('_0000.nii.gz')}
done_pt_uids = existing_pt & existing_ct  # both NIfTIs must exist
# Also skip cases that are already segmented (no need to re-preprocess)
SEG_DIR = f'{DRIVE_ROOT}/segmentations'
os.makedirs(SEG_DIR, exist_ok=True)
already_segmented = {f[:-7] for f in os.listdir(SEG_DIR) if f.endswith('.nii.gz')}
done_pt_uids = done_pt_uids | already_segmented
todo = manifest[~manifest['pt_uid'].isin(done_pt_uids)].copy()
print(f'Total pairs:        {len(manifest)}')
print(f'Already preprocessed: {len(done_pt_uids)}')
print(f'To process:         {len(todo)}')
print(f'Workers:            {NUM_WORKERS}')
print(f'Output:             {PAIRED_DIR}')
print(f'Heartbeat:          {HEARTBEAT}\n', flush=True)

session_done = session_failed = 0
session_start = time.time()
hb('session_start', total_todo=len(todo), num_workers=NUM_WORKERS)

def preprocess_one(idx, total, row):
    tc = time.time()
    try:
        info = build_paired_niftis(row['pt_uid'], row['ct_uid'], PAIRED_DIR)
        elapsed = time.time() - tc
        safe_print(f'  [{idx:4d}/{total}] {row["pt_uid"][-12:]} '
                   f'({elapsed:.0f}s, suvmax={info["suv_max"]:.0f})')
        return ('ok', info)
    except Exception as e:
        with open(ERROR_LOG, 'a') as f:
            f.write(f'PREPROC\t{datetime.now().isoformat()}\t{row["pt_uid"]}\t{e}\n')
        safe_print(f'  [{idx:4d}/{total}] FAILED ({type(e).__name__}: {str(e)[:80]})')
        return ('fail', None)

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    futures = [ex.submit(preprocess_one, i+1, len(todo), row)
               for i, (_, row) in enumerate(todo.iterrows())]
    for fut in as_completed(futures):
        status, _ = fut.result()
        if status == 'ok':
            session_done += 1
        else:
            session_failed += 1
        # Heartbeat every 5 cases
        if (session_done + session_failed) % 5 == 0:
            hb('preprocessing', done=session_done, failed=session_failed,
               todo_remaining=len(todo) - session_done - session_failed,
               elapsed_min=(time.time() - session_start) / 60)

elapsed_hr = (time.time() - session_start) / 3600
hb('finished', session_done=session_done, session_failed=session_failed)
print(f'\n=== Preprocessing done in {elapsed_hr:.2f} hr ===')
print(f'  Preprocessed this session: {session_done}')
print(f'  Failed this session:       {session_failed}')
print(f'  Total paired NIfTIs on Drive: '
      f'{sum(1 for f in os.listdir(PAIRED_DIR) if f.endswith("_0001.nii.gz"))}')
print(f'  Re-run this cell to continue (skips done cases).')

In [ ]:
# Step 6: Verify -- confirm all manifest pairs have paired NIfTIs on Drive
import os
n_pt = sum(1 for f in os.listdir(PAIRED_DIR) if f.endswith('_0001.nii.gz'))
n_ct = sum(1 for f in os.listdir(PAIRED_DIR) if f.endswith('_0000.nii.gz'))
missing = set(manifest['pt_uid']) - {
    f[:-len('_0001.nii.gz')] for f in os.listdir(PAIRED_DIR) if f.endswith('_0001.nii.gz')
}
print(f'Pairs in manifest:     {len(manifest)}')
print(f'CT NIfTIs (_0000):     {n_ct}')
print(f'PET NIfTIs (_0001):    {n_pt}')
print(f'Missing pt_uids:       {len(missing)}')
if missing:
    print(f'  first 5: {sorted(missing)[:5]}')
    print(f'\nRe-run Step 5 to retry, or check {ERROR_LOG} for failures.')
else:
    print(f'\nAll cases preprocessed. Switch runtime to GPU and run infer_autopet_iii.ipynb.')